# PoC — Recuperación densa multilingüe sobre el corpus de quechua wankaDocumento 6 · Taller de Proyectos 1 · Universidad ContinentalEste cuaderno ejecuta el **experimento E5** de la prueba de concepto: la recuperaciónmediante representaciones vectoriales densas multilingües, que no puede ejecutarse en elequipo local por el tamaño de los modelos.Los experimentos E1 a E4 (recuperación léxica) ya se ejecutaron localmente; sus resultadosse comparan aquí con los de este cuaderno.**Entrada:** `fragmentos_v2.jsonl` (corpus segmentado, con documento y página) y`evaluacion_v2.json` (conjunto de evaluación: A, B, C y D).**Salida:** un bloque JSON con las métricas, para incorporarlo al documento.

In [ ]:
#@title 1. Dependencias!pip -q install sentence-transformers==3.* 2>/dev/null || !pip -q install sentence-transformersimport torch, json, re, unicodedata, timeimport numpy as npprint('GPU disponible:', torch.cuda.is_available())

In [ ]:
#@title 2. Cargar el corpus y el conjunto de evaluaciónfrom google.colab import filesprint('Sube fragmentos_v2.jsonl y evaluacion_v2.json')subidos = files.upload()FRAG = [json.loads(l) for l in open('fragmentos_v2.jsonl', encoding='utf-8')]EVAL = json.load(open('evaluacion_v2.json', encoding='utf-8'))IDS = [f['id'] for f in FRAG]print('fragmentos:', len(FRAG), ' consultas:', len(EVAL))

In [ ]:
#@title 3. Normalización de la consulta (misma que en el experimento local)RUIDO = re.compile(r'\b(en|el|la|del|de|quechua|wanka|huanca|variedad|como|se|dice|que|es|'                   r'word|for|in|how|do|you|say|the|necesito|termino|para|referirme|estoy|'                   r'buscando|palabra|designa|what|a)\b')def norm(s):    s = unicodedata.normalize('NFD', s.lower())    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')    return re.sub(r'\s+', ' ', s)def depurar(q):    return re.sub(r'[^a-z0-9 ]', ' ', RUIDO.sub(' ', norm(q))).strip()

In [ ]:
#@title 4. Codificación del corpusMODELO = 'intfloat/multilingual-e5-small'  #@param ['intfloat/multilingual-e5-small', 'BAAI/bge-m3']DEPURAR_CONSULTA = True  #@param {type:'boolean'}from sentence_transformers import SentenceTransformermodelo = SentenceTransformer(MODELO)pref_doc = 'passage: ' if 'e5' in MODELO else ''pref_con = 'query: '   if 'e5' in MODELO else ''t0 = time.time()E_doc = modelo.encode([pref_doc + f['texto'] for f in FRAG],                      batch_size=64, normalize_embeddings=True,                      show_progress_bar=True, convert_to_numpy=True)print(f'corpus codificado en {time.time()-t0:.1f} s · dimensión {E_doc.shape}')

In [ ]:
#@title 5. Recuperación y métricasconsultas = [(depurar(e['consulta']) if DEPURAR_CONSULTA else e['consulta']) for e in EVAL]t0 = time.time()E_con = modelo.encode([pref_con + c for c in consultas], batch_size=64,                      normalize_embeddings=True, convert_to_numpy=True)lat_ms = (time.time() - t0) * 1000 / len(consultas)S = E_con @ E_doc.Tdetalle = []for i, e in enumerate(EVAL):    orden = np.argsort(-S[i])[:10]    oro = set(e['oro'])    rank = next((j + 1 for j, k in enumerate(orden) if IDS[k] in oro), None)    detalle.append({'sub': e['subconjunto'], 'rank': rank, 'top1': float(S[i][orden[0]])})def met(sub):    d = [x for x in detalle if x['sub'] == sub]    if not d: return {}    return {'n': len(d),            'recall@1': round(sum(1 for x in d if x['rank'] == 1)/len(d), 4),            'recall@5': round(sum(1 for x in d if x['rank'] and x['rank'] <= 5)/len(d), 4),            'MRR@10': round(sum(1/x['rank'] for x in d if x['rank'])/len(d), 4),            'similitud_top1_media': round(float(np.mean([x['top1'] for x in d])), 4)}pos = [x['top1'] for x in detalle if x['sub'] in ('A_lexico_es', 'B_independiente')]neg = [x['top1'] for x in detalle if x['sub'] == 'C_fuera_de_cobertura']curva = []for u in np.arange(0.30, 0.96, 0.01):    vp = sum(1 for s in pos if s >= u); fn = len(pos) - vp    fp = sum(1 for s in neg if s >= u); vn = len(neg) - fp    p = vp/(vp+fp) if vp+fp else 0.0    r = vp/(vp+fn) if vp+fn else 0.0    f1 = 2*p*r/(p+r) if p+r else 0.0    curva.append({'umbral': round(float(u),2), 'VP': vp, 'FP': fp, 'VN': vn, 'FN': fn,                  'precision': round(p,4), 'recall': round(r,4), 'F1': round(f1,4),                  'abstencion_correcta': round(vn/len(neg),4)})mejor = max(curva, key=lambda x: (x['F1'], -x['FP']))sin_fp = [x for x in curva if x['FP'] == 0]resultado = {'modelo': MODELO, 'consulta_depurada': DEPURAR_CONSULTA,             'fragmentos': len(FRAG), 'latencia_consulta_ms': round(lat_ms, 2),             'A_lexico_es': met('A_lexico_es'), 'B_independiente': met('B_independiente'),             'D_ingles': met('D_ingles'), 'C_fuera_de_cobertura': met('C_fuera_de_cobertura'),             'umbral_mejor_F1': mejor,             'umbral_sin_falsos_positivos': min(sin_fp, key=lambda x: x['umbral']) if sin_fp else None,             'similitud_media_con_respaldo': round(float(np.mean(pos)), 4),             'similitud_media_sin_respaldo': round(float(np.mean(neg)), 4),             'curva': curva}print(json.dumps(resultado, ensure_ascii=False, indent=1))

In [ ]:
#@title 6. Guardar el resultado para incorporarlo al documentowith open('resultados_colab.json', 'w', encoding='utf-8') as f:    json.dump(resultado, f, ensure_ascii=False, indent=1)files.download('resultados_colab.json')

## Qué mira este experimento- **A** consultas léxicas en español · **B** las mismas nociones con fraseo independiente ·  **D** consultas formuladas en inglés · **C** consultas deliberadamente fuera de la cobertura del corpus.- La recuperación léxica local resolvió A y B, pero **falla en D**: no dispone de capacidad  translingüe. La pregunta de este cuaderno es si el codificador multilingüe la recupera.- El barrido de umbral usa como positivos A y B, y como negativos C. El punto de operación  del proyecto no es el de mayor F1, sino **el menor umbral sin falsos positivos**: responder  sin respaldo documental es el error que el sistema no puede cometer.